# Stage 05: Data Storage Submission

This notebook demonstrates environment-driven paths, suffix-routed CSV and Parquet storage, reloading, and validation.

In [1]:
import datetime as dt
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.storage import detect_format, read_df, write_df

load_dotenv(PROJECT_ROOT / ".env")
RAW_DIR = PROJECT_ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
PROCESSED_DIR = PROJECT_ROOT / os.getenv("DATA_DIR_PROCESSED", "data/processed")
print("Raw directory:", RAW_DIR.relative_to(PROJECT_ROOT))
print("Processed directory:", PROCESSED_DIR.relative_to(PROJECT_ROOT))

Raw directory: data/raw
Processed directory: data/processed


## 1. Create the Sample DataFrame

A fixed random seed makes this synthetic AAPL dataset reproducible.

In [2]:
rng = np.random.default_rng(42)
row_count = 20
sample_df = pd.DataFrame({
    "date": pd.date_range("2025-01-02", periods=row_count, freq="B"),
    "ticker": pd.Series(["AAPL"] * row_count, dtype="string"),
    "price": np.round(190 + rng.normal(0, 1.5, row_count).cumsum(), 2),
    "volume": rng.integers(40_000_000, 80_000_000, row_count, dtype=np.int64),
})
print("Shape:", sample_df.shape)
print("Dtypes:")
print(sample_df.dtypes)
display(sample_df.head())

Shape: (20, 4)
Dtypes:
date      datetime64[us]
ticker            string
price            float64
volume             int64
dtype: object


,date,ticker,price,volume
0,2025-01-02,AAPL,190.46,46609163
1,2025-01-03,AAPL,188.90,70323509
2,2025-01-06,AAPL,190.02,68020918
3,2025-01-07,AAPL,191.43,54181038
4,2025-01-08,AAPL,188.51,42716800


## 2. Save CSV and Parquet

The same timestamp identifies both representations of this dataset. `write_df` creates missing directories and selects the writer based on the suffix.

In [3]:
run_timestamp = dt.datetime.now().strftime("%Y%m%d-%H%M")
csv_path = RAW_DIR / f"sample_{run_timestamp}.csv"
parquet_path = PROCESSED_DIR / f"sample_{run_timestamp}.parquet"

write_df(sample_df, csv_path)
write_df(sample_df, parquet_path)
print("CSV format detected:", detect_format(csv_path))
print("Parquet format detected:", detect_format(parquet_path))
print("Saved:", csv_path.relative_to(PROJECT_ROOT))
print("Saved:", parquet_path.relative_to(PROJECT_ROOT))

CSV format detected: csv
Parquet format detected: parquet
Saved: data/raw/sample_20260821-1207.csv
Saved: data/processed/sample_20260821-1207.parquet


## 3. Reload and Validate

CSV is reloaded with automatic parsing for a column named `date`. Parquet preserves the date, string, float, and integer dtypes directly.

In [4]:
def validate_loaded(original, reloaded, label):
    checks = {
        "shape_matches": original.shape == reloaded.shape,
        "columns_match": list(original.columns) == list(reloaded.columns),
        "date_is_datetime": pd.api.types.is_datetime64_any_dtype(reloaded["date"]),
        "ticker_is_text": pd.api.types.is_string_dtype(reloaded["ticker"]),
        "price_is_float": pd.api.types.is_float_dtype(reloaded["price"]),
        "volume_is_integer": pd.api.types.is_integer_dtype(reloaded["volume"]),
        "dates_match": original["date"].equals(reloaded["date"]),
        "tickers_match": original["ticker"].astype(str).equals(reloaded["ticker"].astype(str)),
        "prices_match": np.allclose(original["price"], reloaded["price"]),
        "volumes_match": np.array_equal(original["volume"], reloaded["volume"]),
    }
    assert all(checks.values()), f"{label} validation failed: {checks}"
    return pd.Series(checks, name=label)


csv_df = read_df(csv_path)
parquet_df = read_df(parquet_path)
validation_results = pd.concat([
    validate_loaded(sample_df, csv_df, "CSV"),
    validate_loaded(sample_df, parquet_df, "Parquet"),
], axis=1)
display(validation_results)
print("All validation checks passed:", validation_results.to_numpy().all())

,CSV,Parquet
shape_matches,True,True
columns_match,True,True
date_is_datetime,True,True
ticker_is_text,True,True
price_is_float,True,True
volume_is_integer,True,True
dates_match,True,True
tickers_match,True,True
prices_match,True,True
volumes_match,True,True


All validation checks passed: True


## 4. Utility Error Behavior

The utilities reject unsupported suffixes, create parent folders during writes, report missing files clearly, and convert a missing Parquet-engine `ImportError` into an actionable installation message. The successful Parquet round trip above confirms that `pyarrow` is available in this environment.

In [5]:
try:
    detect_format("sample.xlsx")
except ValueError as error:
    print("Unsupported suffix check:", error)

try:
    read_df(PROJECT_ROOT / "data/raw/does_not_exist.csv")
except FileNotFoundError as error:
    print("Missing file check:", error)

Unsupported suffix check: Unsupported file suffix '.xlsx'. Use .csv, .parquet, .pq, or .parq.
Missing file check: Data file does not exist: /Users/kevin/Desktop/NYU/bootcamp/bootcamp_zizhe_zhou/homework/stage5/data/raw/does_not_exist.csv


## Storage Choice Summary

CSV is retained in `data/raw/` because it is transparent and broadly interoperable. Parquet is stored in `data/processed/` because it is compact, efficient for analytical reads, and preserves types. Paths come from `DATA_DIR_RAW` and `DATA_DIR_PROCESSED` in local `.env`; the reusable utility layer makes the rest of the analysis independent of format-specific pandas calls.